## Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

## Load Libraries

In [3]:
library(UpSetR)
library(openxlsx)
library(readxl)
library(dplyr)
library(glue)
library(ggplot2)
library(viridis)
library(ggrepel)

## Visualize DEG Analysis

### Read in Data

In [4]:
# List all DEG files
files <- list.files(path = "05_DEGs", pattern = "_results.csv$", full.names = TRUE, recursive = TRUE)

# View
head(files)

[1] "05_DEGs/MUT_Guide6_vs_MUT_NonEffector_results.csv"
[2] "05_DEGs/MUT_Guide6_vs_MUT_NT_results.csv"         
[3] "05_DEGs/MUT_Guide6_vs_WT_NT_results.csv"          
[4] "05_DEGs/MUT_NonEffector_vs_MUT_NT_results.csv"    
[5] "05_DEGs/MUT_NonEffector_vs_WT_NT_results.csv"     
[6] "05_DEGs/MUT_NT_vs_WT_NT_results.csv"

### Separate Upregulated and Downregulated Genes

In [5]:
# Save upregulated and downregulated gene lists
for (file in files) {
  # Extract the base filename without extension
  filename <- tools::file_path_sans_ext(basename(file))
  
  # Split the filename into parts based on '_vs_'
  split_name <- unlist(strsplit(filename, "_vs_"))
  
  # Handle different naming conventions and assign group1 and group2
  if (length(split_name) == 2) {
    group1 <- tolower(split_name[1])  
    group2 <- tolower(gsub("_results", "", split_name[2]))  
  }

  # Read the current file
  df <- read.csv(file)
  colnames(df)[1] <- "gene"

  # Filter to only significant DEGs
  df <- df[df$adj.P.Val < 0.01, ]

  # Filter for only upregulated genes
  upregulated_genes <- df[df$logFC > 0, "gene"]

  # Filter for only downregulated genes
  downregulated_genes <- df[df$logFC < 0, "gene"]

  # Write the upregulated genes to a file in the output directory
  upregulated_file <- file.path(paste0("05_DEGs/", group1, "_vs_", group2, "_upregulated.txt"))
  write.table(upregulated_genes, file = upregulated_file, row.names = FALSE, col.names = FALSE, quote = FALSE)

  # Write the downregulated genes to a file in the output directory
  downregulated_file <- file.path(paste0("05_DEGs/", group1, "_vs_", group2, "_downregulated.txt"))
  write.table(downregulated_genes, file = downregulated_file, row.names = FALSE, col.names = FALSE, quote = FALSE)
  
  # Print status
  cat("Processed", filename, "- Up:", length(upregulated_genes), "Down:", length(downregulated_genes), "\n")
}

Processed MUT_Guide6_vs_MUT_NonEffector_results - Up: 2591 Down: 2532 
Processed MUT_Guide6_vs_MUT_NT_results - Up: 3459 Down: 2887 
Processed MUT_Guide6_vs_WT_NT_results - Up: 3752 Down: 3780 
Processed MUT_NonEffector_vs_MUT_NT_results - Up: 2763 Down: 2777 
Processed MUT_NonEffector_vs_WT_NT_results - Up: 3360 Down: 3453 
Processed MUT_NT_vs_WT_NT_results - Up: 2000 Down: 2334 


### Make Upset Plot

In [13]:
# Read in gene list files
files <- list.files(path = "05_DEGs", pattern = "_(upregulated|downregulated)\\.txt$", full.names = TRUE)

# Create label mapping function
create_label <- function(filename) {
  name_without_ext <- tools::file_path_sans_ext(basename(filename))
  
  label <- name_without_ext %>%
    gsub("_downregulated$", " Down", .) %>% 
    gsub("_upregulated$", " Up", .) %>%
    gsub("mut", "MUT", .) %>%
    gsub("nt", "NT", .) %>%
    gsub("wt", "WT", .) %>%
    gsub("noneffector", "Noneffector", .) %>%
    gsub("guide", "Guide ", .) %>%
    gsub("_", " ", .)
  
  return(trimws(label))
}

label_mapping <- setNames(
  sapply(files, create_label),
  tools::file_path_sans_ext(basename(files))
)

In [14]:
# Load genes 
load_gene_lists <- function(file_vector, label_map) {
  gene_list <- list()
  
  for (file in file_vector) {
    if (file.exists(file) && file.info(file)$size > 0) {
      tryCatch({
        gene_names <- read.table(file, stringsAsFactors = FALSE, header = FALSE)$V1
        gene_names <- gene_names[!is.na(gene_names) & gene_names != ""]
        
        if (length(gene_names) > 0) {
          list_name <- tools::file_path_sans_ext(basename(file))
          gene_list[[list_name]] <- unique(gene_names)
          message(paste("Loaded", length(gene_names), "genes from", basename(file)))
        } else {
          message(paste("No valid genes found in:", basename(file)))
        }
      }, error = function(e) {
        message(paste("Error reading file:", basename(file), "-", e$message))
      })
    } else {
      message(paste("Skipping empty or non-existent file:", basename(file)))
    }
  }
  return(gene_list)
}

In [15]:
# Load gene lists 
gene_list <- load_gene_lists(files, label_mapping)

Loaded 2532 genes from mut_guide6_vs_mut_noneffector_downregulated.txt

Loaded 2591 genes from mut_guide6_vs_mut_noneffector_upregulated.txt

Loaded 2887 genes from mut_guide6_vs_mut_nt_downregulated.txt

Loaded 3459 genes from mut_guide6_vs_mut_nt_upregulated.txt

Loaded 3780 genes from mut_guide6_vs_wt_nt_downregulated.txt

Loaded 3752 genes from mut_guide6_vs_wt_nt_upregulated.txt

Loaded 2777 genes from mut_noneffector_vs_mut_nt_downregulated.txt

Loaded 2763 genes from mut_noneffector_vs_mut_nt_upregulated.txt

Loaded 3453 genes from mut_noneffector_vs_wt_nt_downregulated.txt

Loaded 3360 genes from mut_noneffector_vs_wt_nt_upregulated.txt

Loaded 2334 genes from mut_nt_vs_wt_nt_downregulated.txt

Loaded 2000 genes from mut_nt_vs_wt_nt_upregulated.txt



In [19]:
# Create output directory
dir.create("00_FIGURES/UpSet_Plots", recursive = TRUE, showWarnings = FALSE)

# UpSet plot
create_upset_plot <- function(gene_list, label_mapping, output_dir = "00_FIGURES/UpSet_Plots") {
  
  if (length(gene_list) == 0) {
    message(paste("No gene lists available - skipping plot"))
    return(NULL)
  }
  
  # Convert the gene list to a data frame suitable for UpSetR
  gene_df <- data.frame(
    Gene = unlist(gene_list),
    Group = rep(names(gene_list), times = sapply(gene_list, length))
  )
  
  # Create a binary presence-absence matrix for UpSetR
  gene_matrix <- as.data.frame.matrix(table(gene_df))
  
  # Rename columns using label mapping if available
  if (!is.null(label_mapping) && length(label_mapping) > 0) {
    common_cols <- intersect(colnames(gene_matrix), names(label_mapping))
    if (length(common_cols) > 0) {
      colnames(gene_matrix)[colnames(gene_matrix) %in% common_cols] <- label_mapping[common_cols]
    }
  }
  
  # Define output filename
  output_file <- file.path(output_dir, paste0("upset_plot_of_DEGs.pdf"))
  
  # Create and save the plot
  pdf(output_file, width = 17, height = 13)
  
  p <- upset(gene_matrix, 
             sets = colnames(gene_matrix), 
             keep.order = TRUE,
             main.bar.color = "steelblue", 
             sets.bar.color = "darkgreen",
             order.by = "freq", 
             text.scale = 0.75,
             mainbar.y.label = "Number of Genes",
             sets.x.label = "Genes per Set")
  
  print(p)
  dev.off()
  
  message(paste("Saved UpSet plot:", output_file))
  return(list(plot = p, gene_matrix = gene_matrix))
}

In [20]:
# Generate plot
upset_result <- create_upset_plot(
  gene_list = gene_list,
  label_mapping = label_mapping
)

gene_matrix <- upset_result$gene_matrix
upset_plot <- upset_result$plot

Saved UpSet plot: 00_FIGURES/UpSet_Plots/upset_plot_of_DEGs.pdf



### View Significantly Overlapping Genes

In [22]:
if (nrow(gene_matrix) > 0 && ncol(gene_matrix) > 0) {
  
  gene_membership <- list()
  for (i in 1:nrow(gene_matrix)) {
    gene_name <- rownames(gene_matrix)[i]
    present_conditions <- colnames(gene_matrix)[gene_matrix[i, ] == 1]
    if (length(present_conditions) > 0) {
      gene_membership[[gene_name]] <- present_conditions
    }
  }
  
  overlap_df <- data.frame(
    Gene = names(gene_membership),
    Condition_Count = sapply(gene_membership, length),
    Conditions = sapply(gene_membership, function(x) paste(sort(x), collapse = ", ")),
    stringsAsFactors = FALSE
  )
  
  overlap_df <- overlap_df[order(-overlap_df$Condition_Count, overlap_df$Gene), ]
  
  sink("00_TABLES/gene_overlap_report.txt")
  cat("GENE OVERLAP ANALYSIS REPORT\n")
  cat("============================\n\n")
  cat("Total unique genes:", length(gene_membership), "\n")
  cat("Total conditions:", ncol(gene_matrix), "\n\n")
  cat("CONDITIONS ANALYZED:\n")
  cat(paste(colnames(gene_matrix), collapse = ", "), "\n\n")
  cat("MOST OVERLAPPED GENES\n=====================\n\n")
  
  current_count <- -1
  for (i in 1:nrow(overlap_df)) {
    if (overlap_df$Condition_Count[i] != current_count) {
      cat("\n", paste(rep("=", 60), collapse = ""), "\n", sep = "")
      cat("GENES APPEARING IN", overlap_df$Condition_Count[i], "CONDITIONS\n")
      cat(paste(rep("=", 60), collapse = ""), "\n\n", sep = "")
      current_count <- overlap_df$Condition_Count[i]
    }
    cat(overlap_df$Gene[i], "\n")
    cat("  Conditions: ", overlap_df$Conditions[i], "\n\n", sep = "")
  }
  sink()
  
  write.csv(overlap_df, "00_TABLES/gene_overlap_report.csv", row.names = FALSE)
  message("Saved overlap report and CSV")
}

Saved overlap report and CSV

